In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from tdc.single_pred import HTS, ADME
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit import DataStructs

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    mean_absolute_error, r2_score
)

In [2]:
# Download the data for classification (HTS) and regression (ADME)
HIV_data = HTS(name = 'HIV') #https://tdcommons.ai/single_pred_tasks/hts/?utm_source#hiv:~:text=BY%204.0.-,HIV,benchmark%20for%20molecular%20machine%20learning.%E2%80%9D%20Chemical%20science%209.2%20(2018)%3A%20513%2D530.,-Dataset%20License%3A 

HIV_df = HIV_data.get_data()

print(f"Shape of HIV data: {HIV_df.shape}")

display(HIV_df.head())
# Y in HIV (classification is a regression label: )

Downloading...
100%|██████████| 2.59M/2.59M [00:00<00:00, 12.1MiB/s]
Loading...
Done!


Shape of HIV data: (41127, 3)


,Drug_ID,Drug,Y
0,Drug 0,CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)...,0
1,Drug 1,C(=Cc1ccccc1)C1=[O+][Cu-3]2([O+]=C(C=Cc3ccccc3...,0
2,Drug 2,CC(=O)N1c2ccccc2Sc2c1ccc1ccccc21,0
3,Drug 3,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,0
4,Drug 4,O=S(=O)(O)CCS(=O)(=O)O,0


In [3]:
# Identify the files and label the columsn
def infer_columns(df):
    smiles_col = None
    label_col = None

    for c in df.columns:
        if c.lower() in ["drug", "smiles", "molecule"]:
            smiles_col = c
        if c.lower() in ["y", "label", "target"]:
            label_col = c

    return smiles_col, label_col

hiv_smiles_col, hiv_label_col = infer_columns(HIV_df)

print("HIV:", hiv_smiles_col, hiv_label_col)

HIV: Drug Y


In [4]:
# Check SMILES with RDKit
def parse_smiles(smiles):
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

hiv_mols = HIV_df[hiv_smiles_col].apply(parse_smiles)

print("Valid HIV molecules:", hiv_mols.notna().sum(), "/", len(hiv_mols))

hiv_invalid = HIV_df.loc[hiv_mols.isna(), [hiv_smiles_col, hiv_label_col]].head()

display(hiv_invalid)

[17:47:52] Explicit valence for atom # 3 Al, 6, is greater than permitted
[17:47:53] Explicit valence for atom # 5 B, 5, is greater than permitted
[17:47:55] Explicit valence for atom # 16 Al, 9, is greater than permitted
[17:47:56] Explicit valence for atom # 4 Al, 9, is greater than permitted
[17:47:58] Explicit valence for atom # 12 Al, 7, is greater than permitted
[17:47:58] Explicit valence for atom # 13 Al, 7, is greater than permitted
[17:47:59] WARNING: not removing hydrogen atom without neighbors
[17:47:59] WARNING: not removing hydrogen atom without neighbors
[17:47:59] Explicit valence for atom # 8 Ge, 5, is greater than permitted


Valid HIV molecules: 41120 / 41127


,Drug,Y
137,O=C1O[Al]23(OC1=O)(OC(=O)C(=O)O2)OC(=O)C(=O)O3,0
987,Cc1ccc([B-2]2(c3ccc(C)cc3)=NCCO2)cc1,0
12882,Oc1ccc(C2Oc3cc(O)cc4c3C(=[O+][AlH3-3]35([O+]=C...,0
18293,CC1=C2[OH+][AlH3-3]34([O+]=C2C=CN1C)([O+]=C1C=...,0
30784,CC(c1cccs1)=[N+]1[N-]C(N)=[S+][AlH3-]12[OH+]B(...,0


In [6]:
#Generate fingerprints:
def morgan_fp_from_mol(mol, radius=2, n_bits=2048): # 2048 column-long fingerprint embeddings
    if mol is None:
        return None
    # https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html#:~:text=rdkit.Chem.rdMolDescriptors.GetMorganFingerprintAsBitVect,api%3A%3Aobject%3DNone%20%5B%2Cbool%3DFalse%5D%5D%5D%5D%5D%5D%5D%5D)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    # https://www.rdkit.org/docs/cppapi/namespaceRDKit_1_1MorganFingerprints.html although MorganFingerprintAsBitVect might be deprecated
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr
# https://rdkit.org/docs/GettingStartedInPython.html

# extract the dataframe matrices abd get X, y, clean_df matrices for analyses
def featurize_dataframe(df, smiles_col, label_col, n_bits=2048):
    mols = df[smiles_col].apply(parse_smiles)
    fps = mols.apply(lambda m: morgan_fp_from_mol(m, n_bits=n_bits))

    mask = fps.notna() & df[label_col].notna()
    X = np.vstack(fps[mask].values)
    y = df.loc[mask, label_col].values
    clean_df = df.loc[mask].copy()

    return X, y, clean_df

X_hiv, y_hiv, hiv_clean = featurize_dataframe(HIV_df, hiv_smiles_col, hiv_label_col)
print("HIV fingerprint matrix:", X_hiv.shape, "labels:", y_hiv.shape)

[17:48:34] Explicit valence for atom # 3 Al, 6, is greater than permitted
[17:48:34] Explicit valence for atom # 5 B, 5, is greater than permitted
[17:48:37] Explicit valence for atom # 16 Al, 9, is greater than permitted
[17:48:38] Explicit valence for atom # 4 Al, 9, is greater than permitted
[17:48:41] Explicit valence for atom # 12 Al, 7, is greater than permitted
[17:48:41] Explicit valence for atom # 13 Al, 7, is greater than permitted
[17:48:42] WARNING: not removing hydrogen atom without neighbors
[17:48:42] WARNING: not removing hydrogen atom without neighbors
[17:48:42] Explicit valence for atom # 8 Ge, 5, is greater than permitted
[17:48:43] DEPRECATION WARNING: please use MorganGenerator
[17:48:43] DEPRECATION WARNING: please use MorganGenerator
[17:48:43] DEPRECATION WARNING: please use MorganGenerator
[17:48:43] DEPRECATION WARNING: please use MorganGenerator
[17:48:43] DEPRECATION WARNING: please use MorganGenerator
[17:48:43] DEPRECATION WARNING: please use MorganGenera

HIV fingerprint matrix: (41120, 2048) labels: (41120,)


In [16]:
# Lets train Random Forest on the HIV data and evaluate with cross validation
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# we use roc_auc as scoring metric because its binary classification
# and we want to evaluate the model's ability to rank positive vs negative examples
# prc_auc = area under PRC (Precision vs Recall rate curve)
rf_scores = cross_val_score(rf_model, X_hiv, y_hiv, cv=cv, scoring='average_precision')
print("Random Forest AUC-PR scores:", rf_scores)

Random Forest AUC-PR scores: [0.48750923 0.43145398 0.49205729 0.45995895 0.45396514]


In [ ]:
# Lets do batch passive learning
# active learning
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

def passive(X, y, start_fraction, stop_fraction, size_batch, seed):
    
    ###### Get random 5 (5%) of the labeled data to start with #######
    rand_num_generator = np.random.default_rng(seed) #creates a random number generator with seed
    n = len(y) #number of labeled data points
    permutation_of_indices = rand_num_generator.permutation(n) #random order of indices from 0 to n-1
    starting_obs = int(start_fraction * n) #get number of starting obsvervations
    ending_obs = int(stop_fraction * n) #get number of ending observations
    observed_indices = permutation_of_indices[0:starting_obs].tolist() #get the indices of the starting observations
    unobserved_indices = permutation_of_indices[starting_obs:].tolist() #get the indices of the remaining observations
    
    ###### Define SVM Model & Stratified KFold #######
    model = make_pipeline(
        RandomForestClassifier(n_estimators=100, random_state=42)
    )
    ##### Initialize variables to store results #######
    total_points_to_add = ending_obs - starting_obs
    if total_points_to_add % size_batch == 0:
        num_rounds = total_points_to_add // size_batch
    else:
        num_rounds = total_points_to_add // size_batch + 1  # we add size_batch points in each round, plus one for remainder points

    unobserved_accuracy = []

    ###### Train model and evaluate accuracy in each round #######
    for i in range(num_rounds+1):
        if len(unobserved_indices) == 0:
            break
        X_obs = X[observed_indices]
        y_obs = y[observed_indices]

        #train another model on full observed indices
        X_unobs = X[unobserved_indices]
        y_unobs = y[unobserved_indices]
        X_obs = X[observed_indices]
        y_obs = y[observed_indices]
        model.fit(X_obs, y_obs)
        preds = model.predict(X_unobs)
        unobs_score = average_precision_score(y_unobs, preds)
        unobserved_accuracy.append(unobs_score)

        # add size_batch unobserved points randomly to observed indices
        if i < num_rounds: 
            points_to_add = min(size_batch, len(unobserved_indices))
            new_observed_indices = unobserved_indices[:points_to_add]
            observed_indices.extend(new_observed_indices)
            unobserved_indices = unobserved_indices[points_to_add:]

    return np.array(unobserved_accuracy)

In [ ]:
# Lets do batch active learning
# active learning
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

def active_uncertainty(X, y, start_fraction, stop_fraction, size_batch, seed):
    
    ###### Get random 5 (5%) of the labeled data to start with #######
    rand_num_generator = np.random.default_rng(seed) #creates a random number generator with seed
    n = len(y) #number of labeled data points
    permutation_of_indices = rand_num_generator.permutation(n) #random order of indices from 0 to n-1
    starting_obs = int(start_fraction * n) #get number of starting obsvervations
    ending_obs = int(stop_fraction * n) #get number of ending observations
    observed_indices = permutation_of_indices[0:starting_obs].tolist() #get the indices of the starting observations
    unobserved_indices = permutation_of_indices[starting_obs:].tolist() #get the indices of the remaining observations
    
    ###### Define SVM Model & Stratified KFold #######
    model = make_pipeline(
        RandomForestClassifier(n_estimators=100, random_state=42)
    )
    ##### Initialize variables to store results #######
    total_points_to_add = ending_obs - starting_obs
    if total_points_to_add % size_batch == 0:
        num_rounds = total_points_to_add // size_batch
    else:
        num_rounds = total_points_to_add // size_batch + 1  # we add size_batch points in each round, plus one for remainder points

    unobserved_accuracy = []

    ###### Train model and evaluate accuracy in each round #######
    for i in range(num_rounds+1):
        if len(unobserved_indices) == 0:
            break
        X_obs = X[observed_indices]
        y_obs = y[observed_indices]

        #train another model on full observed indices
        X_unobs = X[unobserved_indices]
        y_unobs = y[unobserved_indices]
        X_obs = X[observed_indices]
        y_obs = y[observed_indices]
        model.fit(X_obs, y_obs)
        preds = model.predict(X_unobs)
        unobs_score = average_precision_score(y_unobs, preds)
        unobserved_accuracy.append(unobs_score)

        # add size_batch unobserved points with highest uncertainty (most disagreement among trees in the random forest)
        if i < num_rounds: 
            probs = model.predict_proba(X_unobs)
            entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1) # add small value to avoid log(0)
            uncertain_indices = np.argsort(entropy)[-size_batch:] # get indices of top size_batch most uncertain points
            new_observed_indices = [unobserved_indices[idx] for idx in uncertain_indices]
            observed_indices.extend(new_observed_indices) # add these to observed indices
            unobserved_indices = [idx for idx in unobserved_indices if idx not in new_observed_indices] # remove these from unobserved indices
        
        if i == num_rounds:
            points_to_add = ending_obs - len(observed_indices)
            if points_to_add > 0:
                probs = model.predict_proba(X_unobs)
                entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
                uncertain_indices = np.argsort(entropy)[-points_to_add:]
                new_observed_indices = [unobserved_indices[idx] for idx in uncertain_indices]
                observed_indices.extend(new_observed_indices)
                unobserved_indices = [idx for idx in unobserved_indices if idx not in new_observed_indices]

    return np.array(unobserved_accuracy)


### Testing Various Batch Sizes

In [ ]:
#set seed

def run_one_seed_active_learning(seed, X, y, size_batch):
    unobserved_accuracy = active_uncertainty(
        X,
        y, 
        start_fraction=0.2,
        stop_fraction=0.5,
        seed=seed,
        size_batch=size_batch
    )
    print(f"Seed {seed} Batch Size {size_batch} - Unobserved Accuracy: {unobserved_accuracy}")
    return unobserved_accuracy

def run_one_seed_passive_learning(seed, X, y, size_batch):
    unobserved_accuracy = passive(
        X,
        y, 
        start_fraction=0.2,
        stop_fraction=0.5,
        seed=seed,
        size_batch=size_batch
    )
    print(f"Seed {seed} Batch Size {size_batch} - Unobserved Accuracy: {unobserved_accuracy}")
    return unobserved_accuracy

In [ ]:
#active learning greedy batch selection
from sklearn.calibration import Parallel, delayed


all_unobserved_accuracies = []
all_unobserved_stds = []

X = X_hiv
y = y_hiv   # labels
seeds = [123, 456, 789] # !!!!!!!!!!!!! Add 9 more seeds
# Repeat active learning for different batch sizes and seeds to see which batc size performs best on average across seeds
results_active_uncertainty_batch5 = Parallel(n_jobs=6, prefer="processes", verbose=10)(delayed(run_one_seed_active_learning)(seed, X, y, size_batch=5) for seed in seeds)
results_active_uncertainty_batch10 = Parallel(n_jobs=6, prefer="processes", verbose=10)(delayed(run_one_seed_active_learning)(seed, X, y, size_batch=10) for seed in seeds)
results_active_uncertainty_batch20 = Parallel(n_jobs=6, prefer="processes", verbose=10)(delayed(run_one_seed_active_learning)(seed, X, y, size_batch=20) for seed in seeds)
results_active_uncertainty_batch50 = Parallel(n_jobs=6, prefer="processes", verbose=10)(delayed(run_one_seed_active_learning)(seed, X, y, size_batch=50) for seed in seeds)
results_active_uncertainty_batch100 = Parallel(n_jobs=6, prefer="processes", verbose=10)(delayed(run_one_seed_active_learning)(seed, X, y, size_batch=100) for seed in seeds)
results_active_uncertainty_batch1000 = Parallel(n_jobs=6, prefer="processes", verbose=10)(delayed(run_one_seed_active_learning)(seed, X, y, size_batch=1000) for seed in seeds)

all_unobserved_accuracies_active_uncertainty_batch5 = [r[0] for r in results_active_uncertainty_batch5]
all_unobserved_accuracies_active_uncertainty_batch10 = [r[0] for r in results_active_uncertainty_batch10]
all_unobserved_accuracies_active_uncertainty_batch20 = [r[0] for r in results_active_uncertainty_batch20]
all_unobserved_accuracies_active_uncertainty_batch50 = [r[0] for r in results_active_uncertainty_batch50]
all_unobserved_accuracies_active_uncertainty_batch100 = [r[0] for r in results_active_uncertainty_batch100]
all_unobserved_accuracies_active_uncertainty_batch1000 = [r[0] for r in results_active_uncertainty_batch1000]